# TechArena CTR — Kaggle CPU/GPU experiments

Upload `techarena_kaggle_data.zip` as a **private Kaggle Dataset**, attach it with Add Input, then Run All. Enable a GPU accelerator for CUDA training if available; CPU also works. No Kaggle API token or GitHub login is required.

This notebook tests a new XGBoost expert against the accepted V53 main local baseline. It does not promise 0.834786 or write a competition CSV. Local, adjusted and official scores are different. The dataset contains prepared causal features, so raw CSV upload is unnecessary. Main day6 is checked first, then the unchanged day5 confirmation only after a positive gate.

In [ ]:
DEVICE = "auto"  # auto / cuda / cpu
MODE = "main"    # main (V53 comparison) / weekday (different diagnostic cohort)
RUN_NAME = "ctr-main-001"  # change for a different experiment or device
DATA_ROOT = None  # automatically locate the attached bundle
VERIFY_INPUTS = True
ROUNDS = 700


In [ ]:
import importlib.metadata, subprocess, sys
try:
    version = importlib.metadata.version("xgboost")
except importlib.metadata.PackageNotFoundError:
    version = "0.0"
if tuple(map(int, version.split(".")[:2])) < (3, 0):
    print("Installing XGBoost >=3.0; enable notebook Internet if needed.")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "xgboost>=3.0,<4"])
import xgboost
print("XGBoost version:", xgboost.__version__)


In [ ]:
from pathlib import Path
import zipfile, stat
if DATA_ROOT is None:
    candidates = list(Path("/kaggle/input").rglob("bundle_manifest.json"))
    if not candidates:
        archives = list(Path("/kaggle/input").rglob("techarena_kaggle_data.zip"))
        assert len(archives) == 1, "Attach the prepared TechArena dataset using Add Input."
        destination = Path("/kaggle/working/ctr-input").resolve()
        destination.mkdir(exist_ok=True)
        with zipfile.ZipFile(archives[0]) as bundle:
            for entry in bundle.infolist():
                target = (destination / entry.filename).resolve()
                assert target.is_relative_to(destination), "Unsafe archive path"
                assert not stat.S_ISLNK(entry.external_attr >> 16), "Archive links are unsupported"
            bundle.extractall(destination)
        candidates = [destination / "bundle_manifest.json"]
    assert len(candidates) == 1, "Multiple bundles found: set DATA_ROOT explicitly."
    DATA_ROOT = candidates[0].parent
DATA_ROOT = Path(DATA_ROOT)
print("Data:", DATA_ROOT)


In [ ]:
import importlib.util
spec = importlib.util.spec_from_file_location("ctr_cloud_runner", DATA_ROOT / "runner.py")
runner = importlib.util.module_from_spec(spec)
spec.loader.exec_module(runner)
output = Path("/kaggle/working") / RUN_NAME
print("Selected device:", runner.choose_device(DEVICE)[0])
summary = runner.run(DATA_ROOT, output, device=DEVICE, mode=MODE, rounds=ROUNDS, verify=VERIFY_INPUTS)
summary


In [ ]:
from IPython.display import FileLink, display
archive = runner.archive_results(output, Path("/kaggle/working") / (RUN_NAME + "-results.zip"))
display(FileLink(str(archive)))
print("Download this results ZIP and share summary.json / report.json for review.")


## Reading the result

`history_adjusted_auc` is the main local metric when MODE is main. An ordinary AUC above 0.82 does not satisfy the minimum adjusted-AUC requirement. Weekday mode uses joint history/calendar weighting on a different cohort and cannot replace the main score. GPU/CPU or library differences may change results. Both dates have been reused; confidence intervals exclude model-selection uncertainty. The last confirmed official competition score is V35 = 0.816623.

The results ZIP contains reports, predictions and saved models. It is **not a competition submission**. Do not upload predictions.npz to the competition. A final full-data refit and validated rowid/probability CSV export must follow any accepted improvement. Use Save Version / Run All to retain outputs in Kaggle before ending the session. No secret keys are needed in this notebook.